# 📈 Sistema GEBRA Portfolio v15.0 – Confluência Total (Grafotária Gebra)
**Baseado na especificação definitiva de Análise Grafotária – Módulos: Anatomia, Matemática, Técnica da Confluência**

- Multi‑timeframe (diário, semanal, mensal)
- Detecção completa de padrões (triângulos, retângulo, OCO, duplos/triplos, bandeira/flâmula, xícara com alça, diamante, canais, pivôs Dow)
- Validação de força do candle (corpo ≥70%, fechamento extremo)
- Volume por padrão (1.5× a 2× média)
- Stop loss específico por padrão
- Alvo = projeção da altura do padrão
- Confluência: mínimo 3 sinais independentes + pontuação anatômica
- Filtro de tendência superior (semanal/mensal)
- Logs detalhados, e‑mail e Telegram

In [ ]:
import os

# ==================== CONFIGURAÇÕES DE ACESSO ====================
EMAIL_REMETENTE = os.getenv('TRADING_EMAIL', '')
SENHA_APP = os.getenv('GMAIL_APP_PASSWORD', '')
BRAPI_TOKEN = os.getenv('BRAPI_API_TOKEN', '') or os.getenv('BRAPI_TOKEN', '')
TELEGRAM_TOKEN = os.getenv('TELEGRAM_TOKEN', '')
TELEGRAM_CHAT_ID = os.getenv('TELEGRAM_CHAT_ID', '')

# ==================== PARÂMETROS GLOBAIS (GRAFOTÁRIA GEBRA) ====================
CAPITAL_TOTAL = 100000.0
RISCO_PERCENTUAL_MINIMO = 0.02   # 2% do capital por trade
RISCO_PERCENTUAL_MAXIMO = 0.10   # 10% máximo
PRECO_MINIMO = 4.00
MAX_SETUPS_POR_DIA = 5
MAX_ATIVOS_POR_SETOR = 2
PAYOFF_MINIMO = 3.0

# --- Parâmetros de padrões ---
TOLERANCIA_NIVEL = 0.02          # 2% para topos/fundos iguais
TOLERANCIA_OMBRO = 0.05          # 5% para altura dos ombros no OCO
CORPO_MINIMO_CANDLE = 0.70       # corpo/range >= 70%
FECHAMENTO_EXTREMIDADE = 0.10    # fechamento a 10% da máxima/mínima
VOLUME_MULT_ALTO = 2.0           # para triângulos, OCO, bandeiras, flâmulas, diamante
VOLUME_MULT_MEDIO = 1.5          # para retângulos, duplos/triplos, xícara, canais
VOLUME_MULT_PIVO = 1.3           # para pivôs de Dow
ATR_PERIODOS = 14

# --- Parâmetros específicos de padrões ---
VOLUME_FORMACAO_MAX_MEDIA = 0.5   # volume médio da bandeira/flâmula < 50% do pré-mastro
CUP_TOPO_CORRECAO_MAX = 0.50     # correção máxima de 50% do topo anterior (xícara)
CUP_HANDLE_MAX_DIAS = 14         # alça com até 14 barras
PIVO_PERNA_MIN_PCT = 0.05        # movimento mínimo de 5% para caracterizar impulso

# --- Confluência ---
PONTUACAO_MINIMA_CONFLUENCIA = 75   # mínimo 75 pontos (0-100)
MIN_SINAIS_CONFLUENCIA = 3          # mínimo de 3 sinais independentes
HABILITAR_PULLBACK = False          # aguardar pullback após rompimento? (modo conservador)
VOLUME_MAX_PULLBACK = 0.8
DIAS_MAX_PULLBACK = 5
IFR_MAX_COMPRA = 70
IFR_MIN_COMPRA = 25
IFR_MAX_VENDA = 75
IFR_MIN_VENDA = 30

# --- Filtros de liquidez e dados ---
VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
ADX_MINIMO = 25
CACHE_TICKERS_FILE = "cache_tickers_b3.json"
FALLBACK_TICKERS = ['PETR4', 'VALE3', 'ITUB4', 'BBDC4', 'BBAS3', 'ABEV3', 'WEGE3', 'RADL3', 'SUZB3', 'GGBR4', 'MGLU3', 'VVAR3', 'RENT3', 'RAIL3', 'CCRO3', 'ELET3', 'CPFE3', 'SBSP3', 'SANB11', 'B3SA3', 'JBSS3', 'BRFS3', 'KLBN11', 'EQTL3']
TICKERS_BLOQUEADOS = ['GFSA3.SA', 'ONCO3.SA', 'PMAM3.SA', 'AZTE3.SA', 'RAIZ4.SA', 'BHIA3.SA', 'CASH3.SA', 'LJQQ3.SA', 'RCSL4.SA', 'HBOR3.SA']

# --- Timeframes ---
ANALISAR_DIARIO = True
ANALISAR_SEMANAL = True
ANALISAR_MENSAL = True

# --- Modo teste ---
MODO_TESTE = False
TESTE_TICKERS = ['PETR4', 'VALE3', 'ITUB4', 'BBDC4', 'BBAS3', 'ABEV3', 'WEGE3', 'RADL3', 'GGBR4', 'MGLU3']

# --- Logging ---
ARQUIVO_LOG = "trading_log_v15.json"
ARQUIVO_LOG_DETALHADO = "execucao_detalhada_v15.log"

print("✅ Parâmetros v15.0 carregados (Confluência Total)")
print(f"   Payoff mínimo: {PAYOFF_MINIMO}:1 | Setups: {sum(1 for _ in range(20))}")
print(f"   Confluência: min {MIN_SINAIS_CONFLUENCIA} sinais | pontuação mínima {PONTUACAO_MINIMA_CONFLUENCIA}")
print(f"   Timeframes: {'Diário ' if ANALISAR_DIARIO else ''}{'Semanal ' if ANALISAR_SEMANAL else ''}{'Mensal' if ANALISAR_MENSAL else ''}")
print(f"   BRAPI: {'✅ Configurada' if BRAPI_TOKEN else '❌ Não configurada (usando yfinance)'}")

In [ ]:
# ================ INSTALAÇÃO E IMPORTAÇÕES ================
!pip install yfinance pandas-ta --quiet 2>/dev/null

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime, timedelta
import time, warnings, json, sys, traceback, gc, socket
from collections import Counter

# Tentar importar scipy para detecção robusta de picos/vales
try:
    from scipy.signal import argrelextrema
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print("⚠️ scipy não disponível. Usando detecção manual de pivôs.")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("✅ Bibliotecas carregadas")

In [ ]:
# ================ LOGGER E UTILITÁRIOS ================
class Logger:
    def __init__(self, log_json, log_detalhado):
        self.log_json = log_json
        self.log_detalhado = log_detalhado
        self.t0 = time.time()
        self.tm = {}
        self.buffer = []
        self.max_buffer = 100

    def log(self, msg, nivel="INFO", extra=None):
        ts = datetime.now().strftime("%H:%M:%S")
        linha = f"[{ts}] [{nivel}] {msg}"
        if extra:
            linha += f" | {extra}"
        print(linha)
        if self.log_detalhado:
            self.buffer.append(linha + "\n")
            if len(self.buffer) >= self.max_buffer:
                self._flush()

    def _flush(self):
        if self.log_detalhado and self.buffer:
            try:
                with open(self.log_detalhado, 'a', encoding='utf-8') as f:
                    f.writelines(self.buffer)
                self.buffer.clear()
            except Exception as e:
                print(f"Erro ao gravar log: {e}")

    def warn(self, msg, extra=None):
        self.log(msg, "WARN", extra)

    def error(self, msg, extra=None):
        self.log(msg, "ERRO", extra)

    def inicio(self, etapa):
        self.tm[etapa] = {'ini': time.time()}
        self.log(f"🚀 INÍCIO: {etapa}", "ETAPA")

    def fim(self, etapa, dados=None):
        if etapa in self.tm:
            dur = time.time() - self.tm[etapa]['ini']
            self.tm[etapa]['dur'] = dur
            msg = f"✅ FIM: {etapa} ({dur:.1f}s)"
            if dados:
                msg += " | " + " | ".join(f"{k}:{v}" for k, v in dados.items())
            self.log(msg, "ETAPA")

    def resumo(self):
        self._flush()
        total = time.time() - self.t0
        self.log("\n" + "="*60, "RESUMO")
        self.log(f"⏱️ Tempo total: {total:.1f}s", "RESUMO")
        for etapa, dados in self.tm.items():
            if 'dur' in dados:
                pct = dados['dur']/total*100 if total>0 else 0
                self.log(f"   • {etapa}: {dados['dur']:.1f}s ({pct:.0f}%)", "RESUMO")
        self.log("="*60 + "\n", "RESUMO")

logger = Logger(ARQUIVO_LOG, ARQUIVO_LOG_DETALHADO)

def _log_exc(contexto, e):
    try:
        msg = f"[{contexto}] {type(e).__name__}: {str(e)[:200]}"
        logger.error(msg)
        with open('traceback_errors.log', 'a', encoding='utf-8') as f:
            f.write(f"\n{'='*60}\n{datetime.now()}\n{contexto}\n{str(e)}\n{traceback.format_exc()}")
    except:
        pass

def enviar_telegram(mensagem, parse_mode='HTML'):
    if not TELEGRAM_TOKEN or not TELEGRAM_CHAT_ID:
        return
    try:
        requests.post(f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage",
                      data={'chat_id': TELEGRAM_CHAT_ID, 'text': mensagem, 'parse_mode': parse_mode}, timeout=10)
    except Exception as e:
        _log_exc('Telegram', e)

def verificar_conectividade():
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=5)
        return True
    except OSError:
        return False

print("✅ Logger e utilitários carregados")

In [ ]:
# ================ DOWNLOAD DE DADOS (BRAPI + YFINANCE) ================
def baixar_dados_brapi(tickers_sa, periodo_anos=5, interval='1d'):
    if not BRAPI_TOKEN:
        return {}
    tickers_limpos = [t.replace('.SA', '') for t in tickers_sa]
    resultados = {}
    erros_http = {}
    for i, ticker in enumerate(tickers_limpos):
        if i % 10 == 0:
            logger.log(f"📡 BRAPI ({interval}): {i+1}/{len(tickers_limpos)}")
        url = f"https://brapi.dev/api/quote/{ticker}"
        params = {'range': f'{periodo_anos}y', 'interval': interval, 'fundamental': 'false', 'token': BRAPI_TOKEN}
        for tentativa in range(2):
            try:
                resp = requests.get(url, params=params, timeout=30)
                if resp.status_code == 200:
                    data = resp.json()
                    quotes = data.get('results', [data])
                    if not isinstance(quotes, list):
                        quotes = [quotes]
                    for quote in quotes:
                        tk = quote['symbol'] + '.SA'
                        if 'historicalDataPrice' in quote:
                            df = pd.DataFrame(quote['historicalDataPrice'])
                            df['date'] = pd.to_datetime(df['date'], unit='s')
                            df.set_index('date', inplace=True)
                            df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'}, inplace=True)
                            df = df[['Open','High','Low','Close','Volume']]
                            resultados[tk] = df
                    break
                elif resp.status_code == 429:
                    time.sleep(2 ** (tentativa + 1))
                else:
                    if tentativa == 1:
                        erros_http[resp.status_code] = erros_http.get(resp.status_code, 0) + 1
                    break
            except Exception as e:
                _log_exc(f'BRAPI {ticker}', e)
                break
        time.sleep(0.25)
    if erros_http:
        logger.warn(f"BRAPI erros HTTP: {erros_http}")
    logger.log(f"📦 BRAPI ({interval}): {len(resultados)} tickers")
    return resultados

def baixar_dados_yfinance_v2(tickers_sa, periodo='5y'):
    data = {}
    for i, t in enumerate(tickers_sa):
        if i % 20 == 0:
            logger.log(f"🔄 yfinance: {i+1}/{len(tickers_sa)}")
        try:
            df = yf.Ticker(t).history(period=periodo, auto_adjust=True)
            if df is not None and not df.empty and 'Close' in df.columns:
                df = df[['Open','High','Low','Close','Volume']].copy()
                data[t] = df
        except Exception as e:
            _log_exc(f'yfinance {t}', e)
        time.sleep(0.3)
    logger.log(f"📦 yfinance: {len(data)} tickers")
    return data

def obter_tickers_brapi():
    if not BRAPI_TOKEN:
        return []
    tickers = []
    try:
        url = "https://brapi.dev/api/quote/list"
        headers = {'Authorization': f'Bearer {BRAPI_TOKEN}'}
        page = 1
        while True:
            params = {'limit': 100, 'page': page, 'type': 'stock'}
            resp = requests.get(url, headers=headers, params=params, timeout=15)
            if resp.status_code != 200:
                break
            data = resp.json()
            stocks = data.get('stocks', [])
            if not stocks:
                break
            for s in stocks:
                tickers.append(s['stock'])
            page += 1
            time.sleep(0.2)
        logger.log(f"📋 BRAPI list: {len(tickers)} tickers")
    except Exception as e:
        logger.warn(f"Erro ao obter lista da brapi: {e}")
    return tickers

def obter_tickers_scraping():
    try:
        resp = requests.get("https://www.dadosdemercado.com.br/acoes", timeout=10, headers={'User-Agent':'Mozilla/5.0'})
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, 'html.parser')
        tickers = []
        for row in soup.select('table tbody tr'):
            cells = row.find_all('td')
            if cells and not cells[0].text.strip().startswith('#'):
                tickers.append(cells[0].text.strip().replace('.SA', ''))
        return tickers
    except Exception as e:
        logger.warn(f"Scraping falhou: {e}")
        return []

def obter_tickers_b3():
    if os.path.exists(CACHE_TICKERS_FILE):
        try:
            with open(CACHE_TICKERS_FILE) as f:
                cache = json.load(f)
            if (datetime.now() - datetime.fromisoformat(cache['timestamp'])).total_seconds() / 3600 < 24:
                logger.log(f"📦 Cache tickers ({len(cache['tickers'])} ativos)")
                return cache['tickers']
        except:
            pass
    tickers = []
    if BRAPI_TOKEN:
        tickers = obter_tickers_brapi()
    if not tickers:
        tickers = obter_tickers_scraping()
    if not tickers:
        tickers = FALLBACK_TICKERS.copy()
        logger.warn("Usando lista fallback de tickers")
    if tickers:
        with open(CACHE_TICKERS_FILE, 'w') as f:
            json.dump({'timestamp': datetime.now().isoformat(), 'tickers': tickers}, f)
    return tickers

print("✅ Módulo de download carregado")

In [ ]:
# ================ FUNÇÕES AUXILIARES E INDICADORES ================
def _safe_divide(a, b, default=np.nan):
    return default if b is None or b == 0 or pd.isna(b) else a / b

def calcular_eficiencia_candle(df):
    corpo = abs(df['Close'] - df['Open'])
    sombra_sup = df['High'] - df[['Close', 'Open']].max(axis=1)
    sombra_inf = df[['Close', 'Open']].min(axis=1) - df['Low']
    range_total = df['High'] - df['Low']
    range_total = range_total.replace(0, np.nan)
    ef = pd.Series(index=df.index, dtype=float)
    alta = df['Close'] > df['Open']
    baixa = df['Close'] < df['Open']
    ef[alta] = 1 - (sombra_sup[alta] / range_total[alta])
    ef[baixa] = 1 - (sombra_inf[baixa] / range_total[baixa])
    return ef

def validar_candle_forca(row, direcao='COMPRA'):
    """
    Verifica se o candle tem corpo >= 70% do range e fechamento próximo à extremidade.
    Retorna (bool, str_motivo)
    """
    rng = row['High'] - row['Low']
    if rng <= 0:
        return False, "range zero"
    corpo = abs(row['Close'] - row['Open'])
    if corpo / rng < CORPO_MINIMO_CANDLE:
        return False, f"corpo/range = {corpo/rng:.2f} < {CORPO_MINIMO_CANDLE}"
    if direcao == 'COMPRA':
        fech_rel = (row['Close'] - row['Low']) / rng
        if fech_rel < (1 - FECHAMENTO_EXTREMIDADE):
            return False, f"fechamento a {fech_rel:.0%} da mínima"
    else:  # VENDA
        fech_rel = (row['High'] - row['Close']) / rng
        if fech_rel < (1 - FECHAMENTO_EXTREMIDADE):
            return False, f"fechamento a {fech_rel:.0%} da máxima"
    return True, "OK"

def detectar_picos_via_scipy(series, ordem=5, modo='max'):
    """Retorna lista de (índice, valor) para picos ou vales usando scipy."""
    if not SCIPY_AVAILABLE:
        return []
    if modo == 'max':
        indices = argrelextrema(series.values, np.greater, order=ordem)[0]
    else:
        indices = argrelextrema(series.values, np.less, order=ordem)[0]
    return [(int(i), series.iloc[i]) for i in indices if i > ordem and i < len(series)-ordem]

def detectar_pivos_manuais(series, janela=5, modo='max'):
    """Fallback: pivôs baseados em janela de vizinhança."""
    pivos = []
    for i in range(janela, len(series)-janela):
        if modo == 'max':
            if series.iloc[i] == max(series.iloc[i-janela:i+janela+1]):
                pivos.append((i, series.iloc[i]))
        else:
            if series.iloc[i] == min(series.iloc[i-janela:i+janela+1]):
                pivos.append((i, series.iloc[i]))
    # Filtrar distância mínima (opcional)
    return pivos

def encontrar_picos(series, ordem=5, modo='max'):
    if SCIPY_AVAILABLE:
        return detectar_picos_via_scipy(series, ordem, modo)
    else:
        return detectar_pivos_manuais(series, janela=ordem, modo=modo)

def calcular_fibonacci_retracao(df, n_barras=50):
    if len(df) < n_barras:
        return None
    highs = df['High']
    lows = df['Low']
    max_idx = highs.rolling(n_barras).max().idxmax()
    min_idx = lows.rolling(n_barras).min().idxmin()
    if max_idx < min_idx:
        topo = highs.loc[max_idx:min_idx].max()
        fundo = lows.loc[max_idx:min_idx].min()
    else:
        topo = highs.loc[min_idx:max_idx].max()
        fundo = lows.loc[min_idx:max_idx].min()
    if topo <= fundo or fundo <= 0:
        return None
    diff = topo - fundo
    return {'38.2%': round(topo - diff*0.382, 2),
            '50.0%': round(topo - diff*0.5, 2),
            '61.8%': round(topo - diff*0.618, 2)}

def preco_em_zona_interesse(df, preco, direcao, tolerancia=0.02):
    """Verifica se o preço está próximo de suporte/resistência recente ou nível de Fibonacci."""
    suportes = []
    resistencias = []
    # Pega os últimos 50 pivôs low/high
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-10:]
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-10:]
    for _, val in lows:
        suportes.append(val)
    for _, val in highs:
        resistencias.append(val)
    # Fibonacci
    fib = calcular_fibonacci_retracao(df)
    niveis = []
    if fib:
        niveis.extend([fib['38.2%'], fib['50.0%'], fib['61.8%']])
    # Checar proximidade
    if direcao == 'COMPRA':
        for sup in suportes:
            if abs(preco - sup) / sup <= tolerancia:
                return True
        for niv in niveis:
            if abs(preco - niv) / niv <= tolerancia:
                return True
    else:
        for res in resistencias:
            if abs(res - preco) / preco <= tolerancia:
                return True
        for niv in niveis:
            if abs(niv - preco) / preco <= tolerancia:
                return True
    return False

def verificar_medias_suporte(df, direcao):
    """Verifica alinhamento de médias e se preço está acima/abaixo."""
    if len(df) < 200:
        return False
    mm20 = df['Close'].rolling(20).mean().iloc[-1]
    mm50 = df['Close'].rolling(50).mean().iloc[-1]
    mm200 = df['Close'].rolling(200).mean().iloc[-1]
    preco = df['Close'].iloc[-1]
    if direcao == 'COMPRA':
        return preco > mm20 and mm20 > mm50 and mm50 > mm200
    else:
        return preco < mm20 and mm20 < mm50 and mm50 < mm200

def calcular_atr(df, periodo=ATR_PERIODOS):
    try:
        atr_serie = ta.atr(df['High'], df['Low'], df['Close'], length=periodo)
        if atr_serie is None or atr_serie.empty:
            return 0.0
        return float(atr_serie.iloc[-1])
    except:
        return 0.0

print("✅ Funções auxiliares carregadas")

In [ ]:
# ================ DETECÇÃO DE PADRÕES GRÁFICOS (GEBRA) ================
# (Implementações baseadas na especificação definitiva)

def _mesmo_nivel(vals, tolerancia=TOLERANCIA_NIVEL):
    if len(vals) < 2:
        return True
    ref = vals[0]
    return all(abs(v - ref) / ref <= tolerancia for v in vals)

def detectar_triangulo_simetrico(df, min_pontos=2):
    """Detecta triângulo simétrico: topos descendentes e fundos ascendentes."""
    if len(df) < 30:
        return False, {}
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-min_pontos*2:]
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-min_pontos*2:]
    if len(highs) < min_pontos or len(lows) < min_pontos:
        return False, {}
    topos = [v for _, v in highs[-min_pontos:]]
    fundos = [v for _, v in lows[-min_pontos:]]
    # Topos descendentes? (cada topo < anterior)
    topos_desc = all(topos[i] > topos[i+1] for i in range(len(topos)-1))
    fundos_asc = all(fundos[i] < fundos[i+1] for i in range(len(fundos)-1))
    if topos_desc and fundos_asc:
        altura = max(topos) - min(fundos)
        return True, {'tipo': 'Triângulo Simétrico', 'topos': topos, 'fundos': fundos, 'altura': altura}
    return False, {}

def detectar_triangulo_ascendente(df, min_pontos=2):
    """Topos horizontais, fundos ascendentes."""
    if len(df) < 30:
        return False, {}
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-min_pontos:]
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-min_pontos:]
    if len(highs) < min_pontos or len(lows) < min_pontos:
        return False, {}
    topos = [v for _, v in highs]
    fundos = [v for _, v in lows]
    if _mesmo_nivel(topos) and all(fundos[i] < fundos[i+1] for i in range(len(fundos)-1)):
        altura = max(topos) - min(fundos)
        return True, {'tipo': 'Triângulo Ascendente', 'resistencia': topos[0], 'altura': altura}
    return False, {}

def detectar_triangulo_descendente(df, min_pontos=2):
    """Fundos horizontais, topos descendentes."""
    if len(df) < 30:
        return False, {}
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-min_pontos:]
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-min_pontos:]
    if len(highs) < min_pontos or len(lows) < min_pontos:
        return False, {}
    topos = [v for _, v in highs]
    fundos = [v for _, v in lows]
    if _mesmo_nivel(fundos) and all(topos[i] > topos[i+1] for i in range(len(topos)-1)):
        altura = max(topos) - min(fundos)
        return True, {'tipo': 'Triângulo Descendente', 'suporte': fundos[0], 'altura': altura}
    return False, {}

def detectar_retangulo(df, min_toques=2):
    if len(df) < 20:
        return False, {}
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-min_toques:]
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-min_toques:]
    if len(highs) < min_toques or len(lows) < min_toques:
        return False, {}
    topos = [v for _, v in highs]
    fundos = [v for _, v in lows]
    if _mesmo_nivel(topos) and _mesmo_nivel(fundos):
        altura = topos[0] - fundos[0]
        return True, {'tipo': 'Retângulo', 'resistencia': topos[0], 'suporte': fundos[0], 'altura': altura}
    return False, {}

def detectar_oco(df, tipo='topo'):
    """Detecta OCO topo ou fundo. Retorna (bool, info_dict)."""
    if len(df) < 50:
        return False, {}
    if tipo == 'topo':
        picos = encontrar_picos(df['High'], ordem=5, modo='max')
        if len(picos) < 3:
            return False, {}
        ombro_e, cabeca, ombro_d = picos[-3], picos[-2], picos[-1]
        # Cabeça mais alta que ombros
        if not (cabeca[1] > ombro_e[1] and cabeca[1] > ombro_d[1]):
            return False, {}
        # Ombros em níveis semelhantes (±5%)
        if abs(ombro_e[1] - ombro_d[1]) / ombro_e[1] > TOLERANCIA_OMBRO:
            return False, {}
        # Vales entre eles (neckline)
        vale1 = min(df['Low'].iloc[ombro_e[0]:cabeca[0]])
        vale2 = min(df['Low'].iloc[cabeca[0]:ombro_d[0]])
        neckline = (vale1 + vale2) / 2  # simplificado, poderia ser reta
        altura = cabeca[1] - neckline
        # Validação de volume decrescente (a ser feita externamente)
        return True, {'tipo': 'OCO Topo', 'ombro_esq': ombro_e[1], 'cabeca': cabeca[1], 'ombro_dir': ombro_d[1],
                      'neckline': neckline, 'altura': altura, 'idx_ombro_dir': ombro_d[0]}
    else:  # OCO invertido (fundo)
        vales = encontrar_picos(df['Low'], ordem=5, modo='min')
        if len(vales) < 3:
            return False, {}
        ombro_e, cabeca, ombro_d = vales[-3], vales[-2], vales[-1]
        if not (cabeca[1] < ombro_e[1] and cabeca[1] < ombro_d[1]):
            return False, {}
        if abs(ombro_e[1] - ombro_d[1]) / ombro_e[1] > TOLERANCIA_OMBRO:
            return False, {}
        pico1 = max(df['High'].iloc[ombro_e[0]:cabeca[0]])
        pico2 = max(df['High'].iloc[cabeca[0]:ombro_d[0]])
        neckline = (pico1 + pico2) / 2
        altura = neckline - cabeca[1]
        return True, {'tipo': 'OCO Invertido', 'ombro_esq': ombro_e[1], 'cabeca': cabeca[1], 'ombro_dir': ombro_d[1],
                      'neckline': neckline, 'altura': altura, 'idx_ombro_dir': ombro_d[0]}

def detectar_duplo_topo(df):
    picos = encontrar_picos(df['High'], ordem=5, modo='max')[-2:]
    if len(picos) < 2:
        return False, {}
    t1, t2 = picos[-2], picos[-1]
    if abs(t1[1] - t2[1]) / t1[1] <= TOLERANCIA_NIVEL:
        vale = min(df['Low'].iloc[t1[0]:t2[0]])
        altura = t1[1] - vale
        return True, {'tipo': 'Topo Duplo', 'topo': t1[1], 'vale': vale, 'altura': altura}
    return False, {}

def detectar_duplo_fundo(df):
    vales = encontrar_picos(df['Low'], ordem=5, modo='min')[-2:]
    if len(vales) < 2:
        return False, {}
    f1, f2 = vales[-2], vales[-1]
    if abs(f1[1] - f2[1]) / f1[1] <= TOLERANCIA_NIVEL:
        pico = max(df['High'].iloc[f1[0]:f2[0]])
        altura = pico - f1[1]
        return True, {'tipo': 'Fundo Duplo', 'fundo': f1[1], 'pico': pico, 'altura': altura}
    return False, {}

def detectar_triplo_topo(df):
    picos = encontrar_picos(df['High'], ordem=5, modo='max')[-3:]
    if len(picos) < 3:
        return False, {}
    t1, t2, t3 = picos[-3], picos[-2], picos[-1]
    if all(abs(t[1] - t1[1]) / t1[1] <= TOLERANCIA_NIVEL for t in [t1, t2, t3]):
        vale = min(min(df['Low'].iloc[t1[0]:t2[0]]), min(df['Low'].iloc[t2[0]:t3[0]]))
        altura = t1[1] - vale
        return True, {'tipo': 'Topo Triplo', 'topo': t1[1], 'vale': vale, 'altura': altura}
    return False, {}

def detectar_triplo_fundo(df):
    vales = encontrar_picos(df['Low'], ordem=5, modo='min')[-3:]
    if len(vales) < 3:
        return False, {}
    f1, f2, f3 = vales[-3], vales[-2], vales[-1]
    if all(abs(f[1] - f1[1]) / f1[1] <= TOLERANCIA_NIVEL for f in [f1, f2, f3]):
        pico = max(max(df['High'].iloc[f1[0]:f2[0]]), max(df['High'].iloc[f2[0]:f3[0]]))
        altura = pico - f1[1]
        return True, {'tipo': 'Fundo Triplo', 'fundo': f1[1], 'pico': pico, 'altura': altura}
    return False, {}

def detectar_bandeira(df, max_dias=20):
    """Bandeira de continuação: movimento brusco (mastro) + canal lateral/descendente curto."""
    if len(df) < max_dias + 10:
        return False, {}
    # Mastro: últimos 5 candles antes da bandeira
    preco_inicio = df['Close'].iloc[-max_dias-10]
    preco_fim_mastro = df['Close'].iloc[-max_dias]
    mastro = abs(preco_fim_mastro - preco_inicio)
    if mastro / preco_inicio < 0.05:  # mínimo 5%
        return False, {}
    # Durante a bandeira (últimos max_dias), amplitude reduzida
    alt_max = df['High'].iloc[-max_dias:].max()
    alt_min = df['Low'].iloc[-max_dias:].min()
    if (alt_max - alt_min) > 0.5 * mastro:
        return False, {}
    direcao = 'COMPRA' if preco_fim_mastro > preco_inicio else 'VENDA'
    return True, {'tipo': 'Bandeira', 'mastro': mastro, 'direcao': direcao}

def detectar_flamula(df, max_dias=15):
    """Flâmula: mini triângulo simétrico após mastro."""
    if len(df) < max_dias + 10:
        return False, {}
    # Verifica movimento brusco anterior
    preco_inicio = df['Close'].iloc[-max_dias-10]
    preco_fim_mastro = df['Close'].iloc[-max_dias]
    mastro = abs(preco_fim_mastro - preco_inicio)
    if mastro / preco_inicio < 0.05:
        return False, {}
    # Verifica congestão em forma de triângulo
    highs = df['High'].iloc[-max_dias:]
    lows = df['Low'].iloc[-max_dias:]
    if len(highs) < 10:
        return False, {}
    # Tendência de topos descendentes e fundos ascendentes?
    picos = encontrar_picos(highs, ordem=2, modo='max')
    vales = encontrar_picos(lows, ordem=2, modo='min')
    if len(picos) >= 2 and len(vales) >= 2:
        topos_desc = all(picos[i][1] > picos[i+1][1] for i in range(len(picos)-1))
        fundos_asc = all(vales[i][1] < vales[i+1][1] for i in range(len(vales)-1))
        if topos_desc and fundos_asc:
            direcao = 'COMPRA' if preco_fim_mastro > preco_inicio else 'VENDA'
            return True, {'tipo': 'Flâmula', 'mastro': mastro, 'direcao': direcao}
    return False, {}

def detectar_cup_handle(df, max_alca_dias=14):
    """Xícara com alça: fundo arredondado + consolidação lateral na borda."""
    if len(df) < 60:
        return False, {}
    # Encontrar topo inicial e fundo da xícara
    highs = df['High']
    topo_inicio = highs.iloc[-60:].max()
    idx_topo = highs.idxmax()
    # Fundo após o topo
    apos_topo = df.loc[idx_topo:]
    fundo = apos_topo['Low'].min()
    profundidade = (topo_inicio - fundo) / topo_inicio
    if not (0.3 <= profundidade <= CUP_TOPO_CORRECAO_MAX):
        return False, {}
    # Verifica recuperação próxima ao topo (dentro de 5%)
    recuperacao = df['Close'].iloc[-1]
    if recuperacao < topo_inicio * 0.95:
        return False, {}
    # Alça: consolidação lateral/descendente nos últimos dias
    alca = df.iloc[-max_alca_dias:]
    if len(alca) < 5:
        return False, {}
    amplitude_alca = (alca['High'].max() - alca['Low'].min()) / alca['Low'].min()
    if amplitude_alca > 0.05:
        return False, {}
    return True, {'tipo': 'Cup&Handle', 'profundidade': profundidade, 'altura': topo_inicio - fundo, 'resistencia': topo_inicio}

def detectar_diamante(df, min_barras=40):
    """Diamante: megafone seguido de triângulo."""
    if len(df) < min_barras:
        return False, {}
    # Fase 1: megafone (topos mais altos, fundos mais baixos)
    metade = min_barras // 2
    primeira_metade = df.iloc[-min_barras:-metade]
    picos1 = encontrar_picos(primeira_metade['High'], ordem=3, modo='max')
    vales1 = encontrar_picos(primeira_metade['Low'], ordem=3, modo='min')
    if len(picos1) < 2 or len(vales1) < 2:
        return False, {}
    megafone = (picos1[-1][1] > picos1[0][1]) and (vales1[-1][1] < vales1[0][1])
    if not megafone:
        return False, {}
    # Fase 2: triângulo simétrico
    segunda_metade = df.iloc[-metade:]
    picos2 = encontrar_picos(segunda_metade['High'], ordem=3, modo='max')
    vales2 = encontrar_picos(segunda_metade['Low'], ordem=3, modo='min')
    if len(picos2) < 2 or len(vales2) < 2:
        return False, {}
    triangulo = (picos2[-1][1] < picos2[0][1]) and (vales2[-1][1] > vales2[0][1])
    if triangulo:
        altura = max(df['High'].iloc[-min_barras:]) - min(df['Low'].iloc[-min_barras:])
        return True, {'tipo': 'Diamante', 'altura': altura}
    return False, {}

def detectar_canal_tendencia(df, min_toques=2):
    """Detecta canal de alta ou baixa."""
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-min_toques:]
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-min_toques:]
    if len(highs) >= min_toques and len(lows) >= min_toques:
        # Testa tendência de alta: lows ascendentes
        if all(lows[i][1] < lows[i+1][1] for i in range(len(lows)-1)):
            altura = highs[-1][1] - lows[-1][1]
            return True, {'tipo': 'Canal de Alta', 'direcao': 'COMPRA', 'altura': altura}
        if all(highs[i][1] > highs[i+1][1] for i in range(len(highs)-1)):
            altura = highs[-1][1] - lows[-1][1]
            return True, {'tipo': 'Canal de Baixa', 'direcao': 'VENDA', 'altura': altura}
    return False, {}

def detectar_pivo_dow(df, direcao='COMPRA'):
    """Pivô de alta (Dow): fundo mais alto e rompimento do topo anterior."""
    if len(df) < 30:
        return False, {}
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-3:]
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-3:]
    if len(lows) < 2 or len(highs) < 2:
        return False, {}
    if direcao == 'COMPRA':
        # Fundo mais alto e preço acima do topo anterior
        if lows[-1][1] > lows[-2][1] and df['Close'].iloc[-1] > highs[-2][1]:
            altura = highs[-2][1] - lows[-2][1]
            return True, {'tipo': 'Pivô de Alta', 'altura': altura}
    else:  # VENDA
        if highs[-1][1] < highs[-2][1] and df['Close'].iloc[-1] < lows[-2][1]:
            altura = highs[-2][1] - lows[-2][1]
            return True, {'tipo': 'Pivô de Baixa', 'altura': altura}
    return False, {}

print("✅ Detecção de padrões carregada")

In [ ]:
# ================ MÓDULO DE CONFLUÊNCIA ================
def contar_sinais_confluencia(df, direcao, setup_detectado, volume_rompimento, media_volume_20, rsi_atual, rsi_anterior, mm20, mm50, mm200):
    sinais = 0
    detalhes = []
    # 1. Padrão gráfico
    if setup_detectado:
        sinais += 1
        detalhes.append("padrão_gráfico")
    # 2. Volume pico
    if volume_rompimento >= media_volume_20 * VOLUME_MULT_MEDIO:
        sinais += 1
        detalhes.append("volume_pico")
    # 3. Médias alinhadas OU IFR saindo de extremo
    if direcao == 'COMPRA':
        medias_ok = mm20 > mm50 > mm200 and df['Close'].iloc[-1] > mm20
        if medias_ok:
            sinais += 1
            detalhes.append("medias_alinhavadas")
        elif rsi_anterior < 30 and rsi_atual > 30:
            sinais += 1
            detalhes.append("ifr_saindo_sobrevenda")
    else:
        medias_ok = mm20 < mm50 < mm200 and df['Close'].iloc[-1] < mm20
        if medias_ok:
            sinais += 1
            detalhes.append("medias_alinhavadas")
        elif rsi_anterior > 70 and rsi_atual < 70:
            sinais += 1
            detalhes.append("ifr_saindo_sobrecompra")
    return sinais, detalhes

def verificar_anatomia_confluencia(df, direcao, entrada, info_setup):
    """Pontua as 4 camadas: esqueleto (tendência Dow), músculo (padrão), célula (zona), sistema nervoso (MM+IFR)."""
    score = 0
    camadas_ok = []
    # 1. Esqueleto – tendência Dow (topos/fundos crescentes)
    highs = encontrar_picos(df['High'], ordem=5, modo='max')[-4:]
    lows = encontrar_picos(df['Low'], ordem=5, modo='min')[-4:]
    if len(highs) >= 2 and len(lows) >= 2:
        tendencia_alta = all(highs[i][1] < highs[i+1][1] for i in range(len(highs)-1)) and all(lows[i][1] < lows[i+1][1] for i in range(len(lows)-1))
        tendencia_baixa = all(highs[i][1] > highs[i+1][1] for i in range(len(highs)-1)) and all(lows[i][1] > lows[i+1][1] for i in range(len(lows)-1))
        if (direcao == 'COMPRA' and tendencia_alta) or (direcao == 'VENDA' and tendencia_baixa):
            score += 25
            camadas_ok.append("esqueleto")
    # 2. Músculo – padrão gráfico detectado
    if info_setup and info_setup.get('tipo'):
        score += 25
        camadas_ok.append("musculo")
    # 3. Célula – preço em zona de interesse (suporte/resistência/Fibonacci)
    if preco_em_zona_interesse(df, entrada, direcao):
        score += 25
        camadas_ok.append("celula")
    # 4. Sistema nervoso – MMs suporte e IFR não exaustão
    mm20 = df['Close'].rolling(20).mean().iloc[-1]
    mm50 = df['Close'].rolling(50).mean().iloc[-1]
    mm200 = df['Close'].rolling(200).mean().iloc[-1]
    rsi = ta.rsi(df['Close'], length=14).iloc[-1] if len(df)>=14 else 50
    if direcao == 'COMPRA':
        mm_ok = entrada > mm20 > mm50 > mm200
        ifr_ok = IFR_MIN_COMPRA < rsi < IFR_MAX_COMPRA
    else:
        mm_ok = entrada < mm20 < mm50 < mm200
        ifr_ok = IFR_MIN_VENDA < rsi < IFR_MAX_VENDA
    if mm_ok and ifr_ok:
        score += 25
        camadas_ok.append("sistema_nervoso")
    elif mm_ok or ifr_ok:
        score += 15
    return score, camadas_ok

def validar_confluencia_total(df, direcao, entrada, setup_detectado, info_setup, volume_atual, media_volume_20, rsi_atual, rsi_anterior, mm20, mm50, mm200):
    """Retorna (aprovado, motivo, score, sinais, camadas)."""
    # 1. Contagem de sinais independentes (mínimo 3)
    n_sinais, sinais_det = contar_sinais_confluencia(df, direcao, setup_detectado, volume_atual, media_volume_20, rsi_atual, rsi_anterior, mm20, mm50, mm200)
    if n_sinais < MIN_SINAIS_CONFLUENCIA:
        return False, f"Apenas {n_sinais} de {MIN_SINAIS_CONFLUENCIA} sinais de confluência", 0, [], []
    # 2. Anatomia (pontuação)
    score_anat, camadas = verificar_anatomia_confluencia(df, direcao, entrada, info_setup)
    if score_anat < PONTUACAO_MINIMA_CONFLUENCIA:
        return False, f"Pontuação anatômica {score_anat} < {PONTUACAO_MINIMA_CONFLUENCIA}", score_anat, sinais_det, camadas
    return True, "Confluência total aprovada", score_anat, sinais_det, camadas

print("✅ Módulo de confluência carregado")

In [ ]:
# ================ CÁLCULO DE STOP E ALVO POR PADRÃO ================
def calcular_stop_por_padrao(df, entrada, direcao, setup_nome, info_setup, atr):
    """Stop loss específico conforme anatomia do padrão."""
    if direcao == 'COMPRA':
        if setup_nome in ['Triângulo Simétrico', 'Triângulo Ascendente']:
            ultimo_fundo = info_setup.get('fundos', [None])[-1] if 'fundos' in info_setup else None
            if ultimo_fundo:
                return ultimo_fundo - atr
        elif setup_nome == 'Retângulo':
            suporte = info_setup.get('suporte', 0)
            if suporte:
                return suporte - atr
        elif setup_nome in ['OCO Invertido', 'Fundo Duplo', 'Fundo Triplo']:
            fundo_padrao = info_setup.get('fundo', info_setup.get('cabeca', 0))
            return fundo_padrao - atr
        elif setup_nome in ['Bandeira', 'Flâmula']:
            # Mínimo da bandeira
            return df['Low'].iloc[-min(20, len(df)):].min() - atr
        elif setup_nome == 'Cup&Handle':
            return info_setup.get('resistencia', entrada) * 0.97  # abaixo da alça
        elif setup_nome == 'Canal de Alta':
            return df['Low'].iloc[-min(10, len(df)):].min() - atr
        elif setup_nome == 'Pivô de Alta':
            return df['Low'].iloc[-min(10, len(df)):].min() - atr
        # Fallback genérico
        return entrada - 1.8 * atr
    else:  # VENDA
        if setup_nome in ['Triângulo Descendente', 'Triângulo Simétrico']:
            ultimo_topo = info_setup.get('topos', [None])[-1] if 'topos' in info_setup else None
            if ultimo_topo:
                return ultimo_topo + atr
        elif setup_nome == 'Retângulo':
            resistencia = info_setup.get('resistencia', 0)
            if resistencia:
                return resistencia + atr
        elif setup_nome in ['OCO Topo', 'Topo Duplo', 'Topo Triplo']:
            topo_padrao = info_setup.get('topo', info_setup.get('cabeca', 0))
            return topo_padrao + atr
        elif setup_nome in ['Bandeira', 'Flâmula']:
            return df['High'].iloc[-min(20, len(df)):].max() + atr
        elif setup_nome == 'Canal de Baixa':
            return df['High'].iloc[-min(10, len(df)):].max() + atr
        elif setup_nome == 'Pivô de Baixa':
            return df['High'].iloc[-min(10, len(df)):].max() + atr
        return entrada + 1.8 * atr

def calcular_alvo_por_padrao(entrada, direcao, setup_nome, info_setup):
    """Alvo baseado na altura projetada do padrão."""
    altura = info_setup.get('altura', info_setup.get('mastro', info_setup.get('profundidade', 0)))
    if altura <= 0:
        # Fallback: arriscar 3x o stop (genérico)
        return None
    if direcao == 'COMPRA':
        return entrada + altura
    else:
        return entrada - altura

def calcular_payoff(entrada, alvo, stop, direcao, custos=0.003):
    if direcao == 'COMPRA':
        risco = entrada - stop
        retorno = alvo - entrada
    else:
        risco = stop - entrada
        retorno = entrada - alvo
    if risco <= 0:
        return 0.0
    payoff = retorno / risco
    # Descontar custos
    payoff -= custos / risco
    return round(max(0, payoff), 2)

print("✅ Cálculo de stop/alvo carregado")

In [ ]:
# ================ ANÁLISE PRINCIPAL DE TIMEFRAME ================
def analisar_timeframe(data_dict, nome_tf, tickers_liquidos, tendencia_superior=None):
    oportunidades = []
    status = []
    contagem_setores = {}
    analisados = 0
    
    for ticker in tickers_liquidos:
        df_raw = data_dict.get(ticker)
        if df_raw is None or df_raw.empty:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': 'Sem dados'})
            continue
        
        # Normalização
        df = df_raw.copy()
        df.index = pd.to_datetime(df.index)
        df.sort_index(inplace=True)
        if len(df) < 50:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': 'Poucos dados'})
            continue
        
        analisados += 1
        entrada = df['Close'].iloc[-1]
        if entrada < PRECO_MINIMO:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Preço R$ {entrada:.2f} < {PRECO_MINIMO}'})
            continue
        
        # Indicadores
        atr = calcular_atr(df)
        if atr == 0:
            atr = entrada * 0.02
        mm20 = df['Close'].rolling(20).mean().iloc[-1]
        mm50 = df['Close'].rolling(50).mean().iloc[-1]
        mm200 = df['Close'].rolling(200).mean().iloc[-1] if len(df)>=200 else mm50
        rsi_series = ta.rsi(df['Close'], length=14)
        rsi_atual = rsi_series.iloc[-1] if rsi_series is not None and not rsi_series.empty else 50
        rsi_anterior = rsi_series.iloc[-2] if rsi_series is not None and len(rsi_series)>=2 else rsi_atual
        media_volume = df['Volume'].rolling(20).mean().iloc[-1]
        volume_atual = df['Volume'].iloc[-1]
        
        # Verificar tendência Dow (esqueleto)
        # (já será feito na confluência)
        
        # Detectar padrões (primeiro identificar direção)
        # Tentar padrões de alta e de baixa separadamente
        setup_detectado = False
        info_setup = {}
        direcao = None
        
        # Padrões de COMPRA
        padroes_compra = [
            (detectar_triangulo_ascendente, 'Triângulo Ascendente'),
            (detectar_triangulo_simetrico, 'Triângulo Simétrico'),
            (detectar_retangulo, 'Retângulo'),
            (detectar_duplo_fundo, 'Fundo Duplo'),
            (detectar_triplo_fundo, 'Fundo Triplo'),
            (lambda x: detectar_oco(x, 'fundo'), 'OCO Invertido'),
            (detectar_bandeira, 'Bandeira'),
            (detectar_flamula, 'Flâmula'),
            (detectar_cup_handle, 'Cup&Handle'),
            (detectar_diamante, 'Diamante'),
            (detectar_canal_tendencia, 'Canal de Alta'),
            (detectar_pivo_dow, 'Pivô de Alta')
        ]
        
        for func, nome in padroes_compra:
            ok, info = func(df)
            if ok:
                # Para padrões bilaterais (triângulo simétrico, diamante), verificar direção do último movimento
                if nome == 'Triângulo Simétrico':
                    # Verifica se o fechamento está acima do triângulo (compra) ou abaixo (venda)
                    # Por simplicidade, assumimos que o padrão pode ser de continuação da tendência anterior
                    tendencia_anterior = df['Close'].pct_change(20).iloc[-1] > 0
                    if not tendencia_anterior:
                        continue  # não força compra se tendência anterior é baixa
                if nome == 'Diamante':
                    # Bilateral, verificar rompimento
                    if df['Close'].iloc[-1] > df['Close'].iloc[-5]:
                        direcao = 'COMPRA'
                    else:
                        continue
                else:
                    direcao = 'COMPRA'
                setup_detectado = True
                info_setup = info
                break
        
        # Se não encontrou padrão de compra, tenta padrões de VENDA
        if not setup_detectado:
            padroes_venda = [
                (detectar_triangulo_descendente, 'Triângulo Descendente'),
                (detectar_triangulo_simetrico, 'Triângulo Simétrico'),
                (detectar_retangulo, 'Retângulo'),
                (detectar_duplo_topo, 'Topo Duplo'),
                (detectar_triplo_topo, 'Topo Triplo'),
                (lambda x: detectar_oco(x, 'topo'), 'OCO Topo'),
                (detectar_bandeira, 'Bandeira'),
                (detectar_flamula, 'Flâmula'),
                (detectar_diamante, 'Diamante'),
                (detectar_canal_tendencia, 'Canal de Baixa'),
                (detectar_pivo_dow, 'Pivô de Baixa')
            ]
            for func, nome in padroes_venda:
                ok, info = func(df)
                if ok:
                    if nome == 'Triângulo Simétrico':
                        if df['Close'].pct_change(20).iloc[-1] > 0:
                            continue
                    if nome == 'Diamante':
                        if df['Close'].iloc[-1] < df['Close'].iloc[-5]:
                            direcao = 'VENDA'
                        else:
                            continue
                    else:
                        direcao = 'VENDA'
                    setup_detectado = True
                    info_setup = info
                    break
        
        if not setup_detectado:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': 'Nenhum padrão detectado'})
            continue
        
        # Força do candle de rompimento (último candle)
        ok_candle, msg_candle = validar_candle_forca(df.iloc[-1], direcao)
        if not ok_candle:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Candle sem força: {msg_candle}'})
            continue
        
        # Volume: múltiplo conforme padrão
        if setup_nome in ['Triângulo Simétrico', 'Triângulo Ascendente', 'Triângulo Descendente', 'OCO Topo', 'OCO Invertido', 'Bandeira', 'Flâmula', 'Diamante']:
            vol_necessario = media_volume * VOLUME_MULT_ALTO
        else:
            vol_necessario = media_volume * VOLUME_MULT_MEDIO
        if volume_atual < vol_necessario:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Volume {volume_atual:.0f} < {vol_necessario:.0f}'})
            continue
        
        # Stop e alvo
        stop = calcular_stop_por_padrao(df, entrada, direcao, setup_nome, info_setup, atr)
        alvo = calcular_alvo_por_padrao(entrada, direcao, setup_nome, info_setup)
        if alvo is None:
            # Fallback: 3x risco
            risco = abs(entrada - stop)
            alvo = entrada + 3*risco if direcao == 'COMPRA' else entrada - 3*risco
        
        # Payoff
        payoff = calcular_payoff(entrada, alvo, stop, direcao)
        if payoff < PAYOFF_MINIMO:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Payoff {payoff:.2f} < {PAYOFF_MINIMO}'})
            continue
        
        # Risco percentual sobre capital
        risco_percent = abs(entrada - stop) / entrada
        if risco_percent < RISCO_PERCENTUAL_MINIMO or risco_percent > RISCO_PERCENTUAL_MAXIMO:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Risco {risco_percent*100:.1f}% fora dos limites'})
            continue
        
        # Confluência total
        aprovado, motivo_conf, score_anat, sinais_det, camadas = validar_confluencia_total(
            df, direcao, entrada, True, info_setup, volume_atual, media_volume, rsi_atual, rsi_anterior, mm20, mm50, mm200
        )
        if not aprovado:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': motivo_conf})
            continue
        
        # (Opcional) Verificar tendência superior (higher timeframe)
        if tendencia_superior and ticker in tendencia_superior:
            tend_sup = tendencia_superior[ticker]
            if (direcao == 'COMPRA' and tend_sup == 'BAIXA') or (direcao == 'VENDA' and tend_sup == 'ALTA'):
                status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Tendência {tend_sup} no timeframe superior conflita'})
                continue
        
        # Restrição de setor (simplificada)
        setor = ticker.split('.')[0][:4]  # placeholder
        if contagem_setores.get(setor, 0) >= MAX_ATIVOS_POR_SETOR:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Limite de setor {setor}'})
            continue
        contagem_setores[setor] = contagem_setores.get(setor, 0) + 1
        
        # Score final (baseado na pontuação de anatomia + payoff + volume)
        score = score_anat + min(20, payoff*5) + (10 if volume_atual >= media_volume*2 else 0)
        score = min(100, score)
        
        oportunidade = {
            'Ticker': ticker,
            'Timeframe': nome_tf,
            'Setup': setup_nome,
            'Direcao': direcao,
            'Entrada': round(entrada, 2),
            'Stop Loss': round(stop, 2),
            'Alvo': round(alvo, 2),
            'Payoff': payoff,
            'Score': round(score, 1),
            'Confluencia': {'sinais': len(sinais_det), 'detalhes': sinais_det, 'camadas': camadas},
            'Instrucao': f"[{nome_tf}] {setup_nome} – {direcao} em R$ {entrada:.2f}, stop R$ {stop:.2f}, alvo R$ {alvo:.2f}. Payoff {payoff:.2f}:1"
        }
        oportunidades.append(oportunidade)
        status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '✅ APROVADO', 'Motivo': 'Confluência total'})
    
    return oportunidades, status, analisados

print("✅ Função de análise de timeframe carregada")

In [ ]:
# ================ FLUXO PRINCIPAL ================
logger.inicio("Verificação de Conectividade")
if not verificar_conectividade():
    logger.error("Sem conectividade com a internet!")
    raise SystemExit("Sem conexão com a internet")
logger.fim("Verificação de Conectividade")

logger.inicio("Coleta de Tickers")
if MODO_TESTE:
    tickers_b3 = TESTE_TICKERS.copy()
    logger.log(f"⚠️ MODO TESTE: {len(tickers_b3)} tickers")
else:
    tickers_b3 = obter_tickers_b3()
tickers_sa = [t + ".SA" for t in tickers_b3]
logger.fim("Coleta de Tickers", {'total': len(tickers_b3)})

logger.inicio("Download Histórico (Diário)")
data_d = {}
PERIODO_ANOS = 5
if BRAPI_TOKEN:
    data_d = baixar_dados_brapi(tickers_sa, periodo_anos=PERIODO_ANOS, interval='1d')
if len(data_d) < max(5, len(tickers_sa)*0.5):
    logger.log("Fallback yfinance...")
    data_yf = baixar_dados_yfinance_v2(tickers_sa, periodo=f'{PERIODO_ANOS}y')
    if len(data_yf) > len(data_d):
        data_d = data_yf
logger.fim("Download Histórico", {'sucesso': len(data_d), 'tentados': len(tickers_sa)})

if len(data_d) == 0:
    logger.error("Nenhum dado obtido")
    raise SystemExit("Sem dados para análise")

logger.inicio("Filtro de Liquidez")
tickers_liquidos = []
for t in tickers_sa:
    if t in TICKERS_BLOQUEADOS:
        continue
    df = data_d.get(t)
    if df is None or df.empty:
        continue
    vm = df['Volume'].tail(21).mean()
    pc = df['Close'].iloc[-1]
    if pd.notna(vm) and pd.notna(pc) and vm >= VOLUME_MINIMO_ACAO and (vm * pc) >= VOLUME_FINANCEIRO_MINIMO:
        tickers_liquidos.append(t)
if len(tickers_liquidos) < 10:
    tickers_liquidos = [t+".SA" for t in FALLBACK_TICKERS[:20] if t+".SA" in data_d]
logger.fim("Filtro de Liquidez", {'liquidos': len(tickers_liquidos)})

# Resample para semanais/mensais
def resample_tf(df, freq, min_dias=4):
    if df is None or df.empty:
        return None
    agg = {'Open':'first','High':'max','Low':'min','Close':'last','Volume':'sum'}
    dfr = df.resample(freq, closed='right', label='right').agg(agg)
    dfr = dfr.dropna()
    return dfr if len(dfr) >= min_dias else None

logger.inicio("Preparação Multi-TF")
data_diario = {t: data_d[t].copy() for t in tickers_liquidos if t in data_d}
data_semanal = {t: resample_tf(data_d[t], 'W-FRI') for t in tickers_liquidos if t in data_d}
data_mensal = {t: resample_tf(data_d[t], 'ME') for t in tickers_liquidos if t in data_d}
data_semanal = {k:v for k,v in data_semanal.items() if v is not None}
data_mensal = {k:v for k,v in data_mensal.items() if v is not None}
logger.fim("Preparação Multi-TF", {'diario':len(data_diario), 'semanal':len(data_semanal), 'mensal':len(data_mensal)})

# Tendência superior (para filtrar trades diários)
tendencia_superior = {}
if ANALISAR_SEMANAL and data_semanal:
    for t, dfw in data_semanal.items():
        if len(dfw) >= 30:
            mm20w = dfw['Close'].rolling(20).mean().iloc[-1]
            mm50w = dfw['Close'].rolling(50).mean().iloc[-1]
            if mm20w > mm50w:
                tendencia_superior[t] = 'ALTA'
            elif mm20w < mm50w:
                tendencia_superior[t] = 'BAIXA'
            else:
                tendencia_superior[t] = 'LATERAL'

# Executar análises
todas_oportunidades = []
todos_status = []

if ANALISAR_DIARIO:
    logger.inicio("Análise Diária")
    ops_d, stat_d, _ = analisar_timeframe(data_diario, "Diário", tickers_liquidos, tendencia_superior)
    todas_oportunidades.extend(ops_d)
    todos_status.extend(stat_d)
    logger.fim("Análise Diária", {'aprovados': len(ops_d)})

if ANALISAR_SEMANAL:
    logger.inicio("Análise Semanal")
    ops_s, stat_s, _ = analisar_timeframe(data_semanal, "Semanal", tickers_liquidos)
    todas_oportunidades.extend(ops_s)
    todos_status.extend(stat_s)
    logger.fim("Análise Semanal", {'aprovados': len(ops_s)})

if ANALISAR_MENSAL:
    logger.inicio("Análise Mensal")
    ops_m, stat_m, _ = analisar_timeframe(data_mensal, "Mensal", tickers_liquidos)
    todas_oportunidades.extend(ops_m)
    todos_status.extend(stat_m)
    logger.fim("Análise Mensal", {'aprovados': len(ops_m)})

# Ordenar e limitar
todas_oportunidades = sorted(todas_oportunidades, key=lambda x: x['Score'], reverse=True)[:MAX_SETUPS_POR_DIA]

logger.log(f"🎯 Total oportunidades finais: {len(todas_oportunidades)}", "RESULTADO")

# Geração de relatório
def gerar_relatorio(ops, status):
    linhas = []
    linhas.append("="*80)
    linhas.append(f"RELATÓRIO GEBRA v15.0 – {datetime.now().strftime('%d/%m/%Y %H:%M')}")
    linhas.append(f"Oportunidades: {len(ops)}")
    linhas.append("="*80)
    if ops:
        for i, op in enumerate(ops, 1):
            linhas.append(f"{i}. [{op['Timeframe']}] {op['Ticker']} | {op['Setup']} | {op['Direcao']}")
            linhas.append(f"   Entrada: R$ {op['Entrada']:.2f} | Stop: R$ {op['Stop Loss']:.2f} | Alvo: R$ {op['Alvo']:.2f}")
            linhas.append(f"   Payoff: {op['Payoff']:.2f} | Score: {op['Score']}")
            linhas.append(f"   Confluência: {len(op['Confluencia']['sinais'])} sinais | Camadas: {', '.join(op['Confluencia']['camadas'])}")
            linhas.append("")
    else:
        linhas.append("❌ Nenhuma oportunidade encontrada.")
    linhas.append("--- FIM ---")
    return "\n".join(linhas)

relatorio = gerar_relatorio(todas_oportunidades, todos_status)
with open('relatorio_gebra_v15.txt', 'w', encoding='utf-8') as f:
    f.write(relatorio)

# Envio de e-mail e Telegram
def enviar_email(assunto, corpo):
    if not EMAIL_REMETENTE or not SENHA_APP:
        return
    try:
        msg = MIMEMultipart()
        msg['From'] = EMAIL_REMETENTE
        msg['To'] = EMAIL_REMETENTE
        msg['Subject'] = assunto
        msg.attach(MIMEText(corpo, 'plain'))
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as srv:
            srv.login(EMAIL_REMETENTE, SENHA_APP)
            srv.send_message(msg)
        logger.log("📧 E-mail enviado")
    except Exception as e:
        _log_exc('Email', e)

enviar_email(f"GEBRA v15.0 - {len(todas_oportunidades)} ops - {datetime.now().strftime('%d/%m %H:%M')}", relatorio)

def mensagem_telegram(ops):
    if not ops:
        return "📊 GEBRA v15.0\nNenhuma oportunidade."
    msg = f"🚀 GEBRA v15.0 - {datetime.now().strftime('%d/%m %H:%M')}\n\n"
    for i, op in enumerate(ops[:5], 1):
        msg += f"{i}. <b>[{op['Timeframe']}] {op['Ticker']}</b> | {op['Setup']} | {op['Direcao']}\n"
        msg += f"   Entrada: R$ {op['Entrada']:.2f} | Stop: R$ {op['Stop Loss']:.2f}\n"
        msg += f"   Alvo: R$ {op['Alvo']:.2f} | Payoff: {op['Payoff']:.2f}\n\n"
    return msg

enviar_telegram(mensagem_telegram(todas_oportunidades))

# Log JSON detalhado
try:
    aprovados = [s for s in todos_status if 'APROVADO' in s.get('Status','')]
    recusados = [s for s in todos_status if 'Recusado' in s.get('Status','')]
    log_json = {
        'timestamp': datetime.now().isoformat(),
        'status_final': 'concluido',
        'metricas': {
            'total_analisados': len(todos_status),
            'aprovados': len(aprovados),
            'recusados': len(recusados),
            'oportunidades_finais': len(todas_oportunidades)
        },
        'oportunidades': todas_oportunidades,
        'recusados_sumario': Counter([s.get('Motivo','') for s in recusados]).most_common(10)
    }
    with open('log_gebra_v15.json', 'w', encoding='utf-8') as f:
        json.dump(log_json, f, indent=2, default=str)
    logger.log("📄 Log JSON salvo")
except Exception as e:
    _log_exc('log_json', e)

gc.collect()
logger.resumo()
print("\n✅ Análise concluída. Relatório: relatorio_gebra_v15.txt")
print(f"   Oportunidades: {len(todas_oportunidades)}")